# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Lane 4 — CTR / Engagement Opportunity Scoring.**

I'm picking this lane because the starter data already shows a pattern that's real but not obvious from a single glance: raw CTR is not comparable across pages unless you first control for search position. A page sitting in position 2 and a page sitting in position 18 have completely different *expected* click-through rates for the exact same content quality — so "low CTR" only means something once it's measured relative to a page's own position tier.

That's a good fit for this lane for three reasons:

- **It's narrow enough to finish in 7 weeks.** The output is one thing: a ranked list of visible pages that are under-capturing clicks relative to their tier, with reason codes a reviewer can sanity-check.
- **It's not just "train a model."** The first job is building an honest position-adjusted baseline (a residual against the tier median), because a flat rule like "CTR < 0.5" would just re-discover that deep-tier pages have low CTR — it would flag the wrong pages for the wrong reason. A model only earns its place if it beats that adjusted baseline.
- **The starter data supports it.** Section 3 below shows real numbers: position tier alone moves median CTR from 0.00 to 0.24, and there's a large, well-populated pool of visible, high-impression candidate pages to rank.

I considered Lane 2 (Refresh/Opportunity Scoring, broader — covers staleness, thinness, decline, and CTR all together) and Lane 1 (Signal Analysis, more open-ended EDA). I'm starting narrower with Lane 4 because a tightly scoped, position-adjusted CTR queue is easier to validate honestly and easier to hand a reviewer a single clear action for ("rewrite the title/meta") than a mixed bag of five different reasons a page might need attention. I can widen back into Lane 2 later if the CTR signal turns out to be weak — I'm not locked in until the end of Week 4.

In [2]:
# Quick setup — used to sanity-check the lane choice below.
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f"Starter dataset: {df.shape[0]:,} rows x {df.shape[1]} columns, {df['client_id'].nunique()} distinct clients")

Starter dataset: 30,000 rows x 44 columns, 32 distinct clients


## 2. The question: decision, action, cost of a wrong call

**Search question:** Among visible pages (real search position, meaningful impression volume), which ones are under-capturing clicks relative to what their position tier normally earns — and which of those are worth a title/meta/snippet review first?

**Unit of analysis:** one row = one (client_id, content_id) pair — a single published content item, summarized over its trailing 90-day window (impressions, clicks, CTR, average position, engagement). Not a query, not a day, not a client — a content item.

**Output:** a ranked review queue of candidate pages, each with a CTR-gap score (how far below its own position tier's typical CTR it sits), a reason code, and a confidence label.

**The decision this improves:** which page an SEO/content reviewer with limited weekly capacity should look at first when deciding whether to rewrite a title, meta description, or snippet.

**Who acts, and what they do:** a FlyRank content reviewer (or the client-facing strategist who queues work for them) works down the ranked list and either rewrites the page's title/meta/snippet, flags it for a deeper content review, or marks it "monitor" and moves on. If nobody would act differently after seeing the list, the lane has failed its own point.

**Cost of a wrong call:**
- *False positive* (page ranked high, but the low CTR isn't fixable by a title/meta change — e.g. it's a mismatch of intent, or it's noise from low volume): a reviewer spends real time — probably 20-40 minutes per page — rewriting a title that was never the problem, and likely sees no click lift.
- *False negative* (a genuinely under-capturing, high-impression page never makes the list): real forgone clicks keep leaking every day it sits unreviewed. Given the starter data alone, candidate pages here carry a median of roughly 3,000 impressions in 90 days each (see Section 3) — a missed page at that scale is a real, ongoing cost, not a rounding error.
- Because reviewer time is the scarce resource, precision in the *top* of the queue (precision@20 or precision@50) matters more than overall accuracy — a queue that's 90% right in its top 20 but noisy further down is still very usable; a queue that's only 50% right in its top 20 wastes half of every reviewer's morning.

**Why data or ML can help at all:** a simple universal rule ("flag anything with CTR under some fixed number") silently confuses "this page has a genuine CTR problem" with "this page is in a tier where low CTR is normal." Fixing that requires comparing each page only to its own position-tier peers first — that's a straightforward calculation, not ML. Where data/ML earns its keep is the next layer: once tier is controlled for, the remaining gap likely correlates with several tangled signals at once (content type, intent match, freshness, word count, engagement) in ways too messy to hand-write as a single if-statement. That's the part worth testing a learned ranking against a transparent baseline for — and only keeping if it actually beats that baseline honestly, on held-out clients.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
# 2-3 real numbers backing the Lane 4 (CTR / Engagement Opportunity Scoring) choice.

# 1) Define "visible" pages: real position data + enough impressions to trust the CTR.
#    avg_position == 0 means "no data" in this dataset (see data dictionary), not rank zero.
visible = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 500)].copy()
print(f"Visible pages (avg_position>0 and impressions_90d>=500): {len(visible):,} of {len(df):,} total rows")

# 2) Show that raw CTR is NOT comparable across position tiers -- this is the core justification
#    for a position-adjusted approach instead of a flat "CTR < X" rule.
tier_ctr = visible.groupby('position_tier')['ctr'].median().sort_values(ascending=False)
print("\nMedian CTR by position tier (visible pages only):")
print(tier_ctr)
print(f"\n-> median CTR swings from {tier_ctr.min():.2f} to {tier_ctr.max():.2f} across tiers.")
print("   A single fixed CTR threshold would just re-flag every deep/mid-tier page and miss real")
print("   under-performers sitting in page_1 -- confirming the lane needs tier-adjusted scoring, not a flat rule.")

# 3) Size the opportunity pool: pages below their OWN tier's median CTR (genuine under-capture,
#    not just "low position").
tier_median = visible.groupby('position_tier')['ctr'].transform('median')
visible['below_tier_median'] = visible['ctr'] < tier_median
n_below = int(visible['below_tier_median'].sum())
pct_below = visible['below_tier_median'].mean() * 100
print(f"\nVisible pages below their own tier's median CTR: {n_below:,} of {len(visible):,} ({pct_below:.1f}%)")

# 4) A concrete candidate pool using the guide's low_ctr_visible_page rule, to see the scale of
#    real, actionable impression volume behind this lane.
low_ctr = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)]
print(f"\n'low_ctr_visible_page' candidates (impressions>=500, position 1-20, ctr<0.5): {len(low_ctr):,}")
print(f"Total impressions_90d carried by these candidates: {int(low_ctr['impressions_90d'].sum()):,}")
print(f"Median impressions_90d per candidate: {low_ctr['impressions_90d'].median():,.0f}")

Visible pages (avg_position>0 and impressions_90d>=500): 16,726 of 30,000 total rows

Median CTR by position tier (visible pages only):
position_tier
page_1      0.24
top_3       0.20
striking    0.17
page_3_5    0.09
deep        0.00
Name: ctr, dtype: float64

-> median CTR swings from 0.00 to 0.24 across tiers.
   A single fixed CTR threshold would just re-flag every deep/mid-tier page and miss real
   under-performers sitting in page_1 -- confirming the lane needs tier-adjusted scoring, not a flat rule.

Visible pages below their own tier's median CTR: 7,950 of 16,726 (47.5%)

'low_ctr_visible_page' candidates (impressions>=500, position 1-20, ctr<0.5): 9,759
Total impressions_90d carried by these candidates: 90,968,008
Median impressions_90d per candidate: 3,017


## 4. Careful words: what I can and can't claim

**What this work can claim, when it's done:**
- *Observed* patterns: which measured signals (position tier, impressions, content type, freshness, word count) are associated with a page sitting below its tier's typical CTR, in this 90-day snapshot.
- *Directional* evidence: whether a position-tier-adjusted score does a better job separating likely under-performers from likely fine pages than a flat, un-adjusted rule — measured with precision@K against an honest client-holdout split.
- *Decision-support* output: a ranked, reason-coded review queue that helps a human reviewer spend limited time on the most promising candidates first.

**What this work will never claim:**
- That rewriting a title or meta description *caused* a click increase — that requires an actual experiment (an A/B test or before/after causal design), not an observational ranking.
- That any result reflects how Google's ranking or CTR algorithm actually works internally — I only have observable outcomes (impressions, clicks, position), never the algorithm itself.
- That a "low CTR" page is a guaranteed opportunity — some of the gap will be real noise (low volume), some will be consolidation (a sibling page absorbing the clicks), and some will be a genuine mismatch between search intent and page content that a title rewrite can't fix.
- That results generalize beyond this dataset's 32 pseudonymized clients and its trailing-90-day window — a pattern here is a hypothesis for the client roster and time window in front of me, not a universal SEO law.

I'll use words like *observed*, *associated with*, *this suggests*, and *decision-support* — never *proves*, *causes*, or *guarantees*.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.